![Week 2 Day 1](../assets/banners/w2d1.png)

# Week 2 Day 1 - Why RAG, and Embeddings: Meaning as Geometry

Welcome to week 2! 🎉

Last week you gave GradeWise its first real skill: it read Asha's question paper, marking scheme and methodology and turned them into a validated `ExamPaper` JSON. That's the **"what are we marking?"** half of the problem.

This week is the **"what is the right answer?"** half. A grader that only knows what was in its training data is a grader that guesses. So this week, GradeWise learns the course material itself: the four CS201 PDFs that Asha wrote for her students at Northbridge Institute. By Friday it will have a searchable knowledge base, and, just as importantly, **numbers** that tell us how good that search is.

Today we start with the *why* (hallucinations, knowledge cut-offs and grounding), and then the single idea that makes all of RAG possible: **embeddings**, where meaning becomes geometry.

![Your 6-week roadmap](../assets/roadmap/week2.png)

### 🎯 By the end of this lab you will
1. 🗺️ Know what GradeWise will be able to do by the end of week 2
2. 🤔 Explain *why* we need RAG: hallucinations, knowledge cut-offs and grounding
3. 📐 Turn words into vectors by hand, plot them, and measure similarity with cosine
4. 🧠 Know where real embeddings come from in GradeWise (`gradewise.models.embeddings()`), and why the fake ones we'll use offline are meaningless

### 🎬 Videos for this lab

| # | Lecture |
|:-:|---|
| 1 | Day 1 - Week 2 Kickoff: Recap and Roadmap |
| 2 | Day 1 - Why RAG? Hallucinations, Knowledge Cut-offs and Grounding |
| 3 | ⭐ Day 1 - Embeddings Visually Explained: Meaning as Geometry |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Almost everything in this lab runs <b>offline and free</b>: numpy, matplotlib and the real course PDFs. Only three cells call a real model (the closed-book and open-book answers in Part 2, and the real embeddings in Part 4). No key yet? Those cells will show a friendly <code>MissingAPIKeyError</code>: just read on, nothing later depends on them.</span>
        </td>
    </tr>
</table>

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("ℹ️ OpenAI API Key not set. That's OK for most of this lab: the offline cells use fakes.")
    print("   The few cells that call a real model will tell you so. Setup: Part 5 of the setup guide.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pymupdf
from IPython.display import Markdown, display

from gradewise.config import DATA_DIR, settings

KB = DATA_DIR / "knowledge-base"
print("Knowledge base PDFs:")
for pdf in sorted(KB.glob("*.pdf")):
    print(f"  📄 {pdf.name}")
print(f"\nEmbedding model in your .env: {settings.embedding_model}")

## Part 1: Week 1 recap, and this week's roadmap

🔁 **Recap of week 1:**
- `init_chat_model` and `gradewise.models.chat_model()` give us any model from one `provider:model` string
- Pydantic + structured output turned free text into reliable JSON, with validation and retries
- The exam pack became a validated `ExamPaper`: questions, criteria, marks and methodology

**This week, day by day:**

| Day | What GradeWise learns |
|:-:|---|
| 1 (today) | Why RAG, and embeddings: how meaning becomes geometry |
| 2 | Loading the PDFs well, and cutting them into chunks that respect their structure |
| 3 | A persistent Chroma vector store with metadata filters, and ingestion you can re-run for free |
| 4 | Dense, BM25 and hybrid retrieval, then a first RAG chain that cites its sources |
| 5 | 🧱 Measuring retrieval with MRR and nDCG on 60 test questions, and the simple vs pro pipeline |

Here's the whole week on one diagram. Every box is a day of this week:

![Diagram](../assets/diagrams/86062cbd1645.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
    A[📄 CS201 PDFs] -->|Day 2: load + chunk| B[🧩 Chunks + metadata]
    B -->|Day 1: embed| C[📐 Vectors]
    C -->|Day 3: store| D[(🗄️ Chroma)]
    Q[❓ Question] -->|Day 4: retrieve| D
    D --> E[📚 Top-k chunks]
    E -->|Day 4: generate| F[💬 Answer with citations]
    E -->|Day 5: measure| G[📏 MRR / nDCG]
```

</details>


## Part 2: Why RAG? Hallucinations, knowledge cut-offs and grounding

A large language model is a brilliant generalist with three blind spots that matter enormously for grading:

| Blind spot | What it means | Why Asha cares |
|---|---|---|
| **Knowledge cut-off** | The model only knows what was in its training data, up to some date | Asha's notes were written *this term*. No model has ever seen them |
| **Private knowledge** | It never saw your company's or your course's documents | "CS201 accepts *semafore* for *semaphore*" is a Northbridge rule, not a world fact |
| **Hallucination** | When it doesn't know, it often produces a fluent, confident guess | A confident wrong mark is worse than no mark |

The fix is **grounding**: instead of asking the model to *remember*, we *show* it the relevant material and ask it to answer from that. Finding the right material automatically is **R**etrieval; pasting it into the prompt is **A**ugmentation; the answer is **G**eneration. That's **RAG**.

Think of it as the difference between a **closed-book** and an **open-book** exam. Let's run both.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">The next cell asks a model a question about CS201's <i>own</i> marking rules, with no notes at all. Predict: will it say "I don't know", or will it invent a plausible-sounding policy? Make your guess, then run it.</span>
        </td>
    </tr>
</table>

In [ ]:
from gradewise.models import chat_model

question = (
    "In Northbridge Institute's CS201 course, is a misspelled technical term such as 'semafore' "
    "penalised, and what must a marker do with an illegible answer? Answer in two sentences."
)

llm = chat_model(temperature=0)
closed_book = llm.invoke(question)
display(Markdown("**📕 Closed book:** " + closed_book.text))

Now the open-book version. Today we do the "retrieval" by hand: we open the course handbook PDF ourselves and copy out the page with the marking principles. (Tomorrow we'll load PDFs properly; on Day 4 the retriever will find the right page for us.)

In [ ]:
with pymupdf.open(KB / "cs201_course_handbook.pdf") as handbook:
    pages = [page.get_text() for page in handbook]

# 💡 Our hand-made "retriever": the first page that mentions the marking principles
notes = next(text for text in pages if "General marking principles" in text)
start = notes.index("General marking principles")
excerpt = notes[start : start + 1800]
print(excerpt)

In [ ]:
grounded_prompt = f"""Answer the question using ONLY the course notes below. If they don't cover it, say so.

<notes>
{excerpt}
</notes>

Question: {question}"""

open_book = llm.invoke(grounded_prompt)
display(Markdown("**📗 Open book:** " + open_book.text))

The open-book answer can quote the real rules (principle 1: spelling is not penalised if the meaning is clear; principle 9: illegible answers go to the course leader, not to zero). The closed-book one can only guess, however confident it sounds.

💡 Notice we didn't retrain anything. RAG changes **what the model sees**, not what the model *is*. That's why it's cheap, instant to update (edit a PDF, re-ingest) and auditable: we can show exactly which page an answer came from.

![Diagram](../assets/diagrams/5e95951725fd.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
    Q[❓ Question] --> R[🔎 Retrieve<br/>find relevant chunks]
    R --> A[📎 Augment<br/>paste them into the prompt]
    A --> G[💬 Generate<br/>answer from the notes, with citations]
```

</details>


The hard part is the first box. How do you find "the relevant page" among hundreds, for a question phrased in words that may not appear on that page at all? That's where embeddings come in.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">RAG is the most widely deployed LLM pattern in industry, because almost every organisation's valuable knowledge is private: policies, contracts, product manuals, support tickets, course notes. "Answer from <i>our</i> documents, and show me where it says so" is a request you will hear in every company. For GradeWise it is the difference between "the model thinks this deserves 3/4" and "this deserves 3/4 <b>because</b> page 5 of Unit 1 says…".</span>
        </td>
    </tr>
</table>

## Part 3: Embeddings, meaning as geometry

An **embedding** is a list of numbers (a vector) that represents the *meaning* of a piece of text. The magic property: **texts with similar meanings get vectors that point in similar directions.** Then "find relevant text" becomes "find nearby vectors", which computers do very fast.

Real embedding models use 1,536 or 3,072 dimensions that nobody can interpret. To build intuition, let's invent our own embedding with just **two** dimensions that we *can* read:
- dimension 0: how much a word is about **operating systems**
- dimension 1: how much a word is about **networks**

We'll place some CS201 words by hand.

In [ ]:
# 💡 A hand-made 2-D "embedding": [operating-systems-ness, networks-ness]
toy_2d = {
    "process": [0.90, 0.10],
    "thread": [0.85, 0.15],
    "deadlock": [0.80, 0.05],
    "semaphore": [0.88, 0.08],
    "router": [0.10, 0.90],
    "packet": [0.05, 0.95],
    "TCP": [0.15, 0.85],
    "DNS": [0.10, 0.80],
    "socket": [0.55, 0.60],  # 🧪 an OS object used for networking: it sits in between
}
words = list(toy_2d)
vectors = np.array([toy_2d[w] for w in words])
print(vectors.shape, "→", len(words), "words, 2 dimensions each")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for word, (x, y) in toy_2d.items():
    colour = "#7c5cff" if x > y else "#22d3ee"
    ax.annotate("", xy=(x, y), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=colour, alpha=0.5))
    ax.scatter(x, y, color=colour, s=60)
    ax.text(x + 0.02, y + 0.02, word, fontsize=11)
ax.set_xlim(0, 1.1)
ax.set_ylim(0, 1.1)
ax.set_xlabel("dimension 0: operating systems →")
ax.set_ylabel("dimension 1: networks →")
ax.set_title("A toy 2-D embedding of CS201 words")
ax.grid(alpha=0.3)
plt.show()

Two clusters, exactly as you'd hope: OS words along the bottom-right, networking words up the left, and *socket* in the middle.

### Measuring "nearby": cosine similarity

The standard measure is **cosine similarity**: the cosine of the angle between two vectors.

$$\text{cosine}(a, b) = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$$

It is **1** when the vectors point the same way, **0** when they're at right angles (unrelated) and **−1** when they point in opposite directions. It ignores length, which is what we want: a long paragraph and a short sentence about deadlocks should still count as similar.

In [ ]:
def cosine(a, b) -> float:
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


print(f"deadlock vs semaphore: {cosine(toy_2d['deadlock'], toy_2d['semaphore']):.3f}")
print(f"deadlock vs packet   : {cosine(toy_2d['deadlock'], toy_2d['packet']):.3f}")
print(f"socket   vs TCP      : {cosine(toy_2d['socket'], toy_2d['TCP']):.3f}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Sam types the question <i>"my program froze because two threads are each waiting for the other one"</i>. We'd embed it as roughly <code>[0.9, 0.05]</code>. Before running the next cell, predict the top three words, and whether <i>socket</i> makes the list.</span>
        </td>
    </tr>
</table>

In [ ]:
query = [0.90, 0.05]  # 💡 "two threads each waiting for the other": very OS, not networks

ranked = sorted(words, key=lambda w: cosine(query, toy_2d[w]), reverse=True)
for rank, word in enumerate(ranked, start=1):
    print(f"{rank}. {word:<10} cosine = {cosine(query, toy_2d[word]):.3f}")

🎉 That's semantic search in miniature: Sam never typed the word *deadlock*, but the deadlock vector is right next to his question. **Retrieval = embed the question, then rank the stored vectors by cosine.**

Notice something, though: *every* OS word scores above 0.99. With only two dimensions, all OS words point in almost the same direction, so cosine can barely tell *deadlock* from *thread*. Two dimensions just aren't enough room for meaning.

### A third dimension, and a word with two meanings

Our 2-D world has no room for databases. Real meaning needs many more dimensions, so let's add a third (**databases**) and a tricky word: **lock**, which is both an OS idea (a mutex lock) and a database idea (row locks in transactions).

In [ ]:
# [operating systems, networks, databases]
toy_3d = {
    "process": [0.90, 0.10, 0.05],
    "deadlock": [0.80, 0.05, 0.30],
    "packet": [0.05, 0.95, 0.00],
    "TCP": [0.15, 0.85, 0.05],
    "SQL": [0.05, 0.05, 0.95],
    "transaction": [0.25, 0.05, 0.90],
    "lock": [0.65, 0.05, 0.60],  # 🧪 OS AND databases
}

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(projection="3d")
for word, (x, y, z) in toy_3d.items():
    ax.quiver(0, 0, 0, x, y, z, color="#7c5cff", alpha=0.4, arrow_length_ratio=0.08)
    ax.scatter(x, y, z, color="#7c5cff", s=40)
    ax.text(x, y, z, f" {word}", fontsize=10)
ax.set_xlabel("OS")
ax.set_ylabel("networks")
ax.set_zlabel("databases")
ax.set_title("A toy 3-D embedding")
plt.show()

for other in ("process", "transaction", "packet"):
    print(f"lock vs {other:<12}: {cosine(toy_3d['lock'], toy_3d[other]):.3f}")

*Lock* sits between the OS and database clusters and is far from *packet*. Real embedding models do this in thousands of dimensions, learned from enormous amounts of text, which is why they capture subtleties like "*circular wait*" being about deadlocks.

### Cosine, Euclidean distance or dot product?

Vector databases offer several ways to compare vectors. Here's why we use cosine for text:

In [ ]:
short = np.array(toy_3d["deadlock"])
long = short * 3  # 💡 same direction, 3× longer: think "a whole page about deadlocks"

print(f"cosine(short, long)    = {cosine(short, long):.3f}   ← same meaning: 1.0")
print(f"euclidean(short, long) = {np.linalg.norm(short - long):.3f}   ← looks far apart!")
print(f"dot(short, long)       = {short @ long:.3f}   ← grows with length")
print(f"\ncosine distance = 1 - cosine similarity = {1 - cosine(short, long):.3f}")

💡 Euclidean distance and dot product are both affected by **length**; cosine only by **direction**. Many embedding models (OpenAI's included) return vectors already normalised to length 1, and then all three give the same ranking. Using cosine is the safe default, and it's what GradeWise's Chroma store is configured with on Day 3 (`"hnsw:space": "cosine"`).

## Part 4: Real embeddings come from `gradewise.models.embeddings()`

In GradeWise you never pick an embedding model in code. It comes from `EMBEDDING_MODEL` in your `.env` (by default `openai:text-embedding-3-small`, 1,536 dimensions), through `gradewise.models.embeddings()`, which wraps LangChain's `init_embeddings`.

The next cell embeds a few CS201 sentences for real and squashes the 1,536 dimensions down to 2 with PCA, so we can look at them. (It needs your API key and costs a tiny fraction of a cent.)

In [ ]:
from sklearn.decomposition import PCA

from gradewise.models import embeddings

sentences = [
    "A deadlock needs mutual exclusion, hold and wait, no pre-emption and circular wait.",
    "Two threads each wait forever for a lock the other one holds.",
    "TCP uses a three-way handshake to open a connection.",
    "A router forwards packets between networks.",
    "A transaction must be atomic, consistent, isolated and durable.",
    "Second normal form removes partial dependencies on a composite key.",
]
vecs = np.array(embeddings().embed_documents(sentences))
print("Each sentence became a vector of", vecs.shape[1], "numbers")

points = PCA(n_components=2).fit_transform(vecs)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(points[:, 0], points[:, 1], color="#7c5cff")
for (x, y), text in zip(points, sentences, strict=True):
    ax.text(x, y, " " + text[:38] + "…", fontsize=9)
ax.set_title("Real embeddings, projected to 2-D with PCA")
plt.show()

With a key, you'll see three pairs: the two deadlock sentences together, the two networking ones together and the two database ones together, even though "two threads wait forever" shares almost no words with "circular wait".

### Offline embeddings for this course: fakes that are deterministic but meaningless

From Day 3 on, our offline demos and tests need *some* embedding to fill a vector store without an API key. LangChain ships one: `DeterministicFakeEmbedding`. It turns each text into a random vector **seeded by the text's hash**, so the same text always gives the same vector. That's perfect for testing the plumbing, and useless for meaning.

In [ ]:
from langchain_core.embeddings import DeterministicFakeEmbedding

fake = DeterministicFakeEmbedding(size=64)

a1 = fake.embed_query("deadlock")
a2 = fake.embed_query("deadlock")
b = fake.embed_query("deadlocks")
c = fake.embed_query("SQL joins")

print(f"'deadlock' vs 'deadlock'  : {cosine(a1, a2):+.3f}   ← deterministic: identical")
print(f"'deadlock' vs 'deadlocks' : {cosine(a1, b):+.3f}   ← should be ~1, it's random")
print(f"'deadlock' vs 'SQL joins' : {cosine(a1, c):+.3f}   ← should be low, it's random")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;"><b>Fake embeddings give meaningless similarity.</b> Whenever you see <code>DeterministicFakeEmbedding</code> in this course, the store, filters and pipeline are real but any "nearest neighbour" is luck. Real meaning comes only from <code>gradewise.models.embeddings()</code>. And one rule that applies to real models too: <b>always query a store with the same embedding model that built it</b>. Vectors from different models live in different spaces and can't be compared.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Open <code>gradewise/models.py</code> and find <code>embeddings()</code>: four lines that read <code>EMBEDDING_MODEL</code> from your settings and check the key. Then peek at <code>gradewise/ingestion/store.py</code>: <code>get_vectorstore()</code> uses that same function by default, but lets tests pass <code>DeterministicFakeEmbedding(size=64)</code> instead. That injection is how this whole week runs offline.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- 🗺️ saw the week's plan: PDFs → chunks → vectors → Chroma → retrieval → answers → measurement
- 🤔 met RAG's reason to exist: closed-book models guess, open-book models can quote and cite
- 📐 built 2-D and 3-D embeddings by hand, plotted them and ranked words by cosine similarity
- 🧠 learned that real vectors come from `gradewise.models.embeddings()`, and fake ones only test plumbing

**Tomorrow (Day 2):** we stop copying pages by hand. We'll hold a PDF-loader bake-off on Asha's Unit 1 notes, then compare chunking strategies visually, ending with GradeWise's structure-aware chunker.

⏭ **Next:** `week2/day2.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Grow the toy world. (1) Add <i>mutex</i>, <i>subnet</i> and <i>index</i> to <code>toy_3d</code> and predict each one's nearest neighbour before you compute it. (2) Find a pair of words where Euclidean distance and cosine similarity disagree about which neighbour is closest. (3) Bonus, with a key: add six sentences of your own to the PCA plot and check the clusters. Solutions are in <code>week2/solutions/day1_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• The original RAG paper (Lewis et al., 2020): <a href="https://arxiv.org/abs/2005.11401">arxiv.org/abs/2005.11401</a><br/> • LangChain embeddings: <a href="https://docs.langchain.com/oss/python/langchain/knowledge-base">docs.langchain.com/oss/python/langchain/knowledge-base</a><br/> • OpenAI embeddings guide: <a href="https://platform.openai.com/docs/guides/embeddings">platform.openai.com/docs/guides/embeddings</a></span>
        </td>
    </tr>
</table>